# GateScore-Lipid Usage Tutorial
## Conjugatability-Aware BLNP Module Prioritization Protocol

This notebook demonstrates how to use GateScore-Lipid screening results
to explore and prioritize candidate BLNP modules.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load corrected screening results
df = pd.read_csv('data/lipid_module_round2_corrected_full_screen.csv')
top50 = pd.read_csv('data/lipid_module_top50_corrected.csv')

print(f'Screening results: {len(df):,} molecules')
print(f'Columns: {list(df.columns)}')
print(f'\nTop 50 candidates loaded: {len(top50)}')

## 1. Search-Space Reduction

GateScore-Lipid reduces 9,599 molecules to 462 elevated-gating conjugatable candidates.

In [ ]:
n_total = len(df)
n_conj = (df['n_conjugatable_groups'] > 0).sum()
n_gate50 = (df['S_gating'] > 50).sum()
n_elevated = ((df['S_gating'] > 50) & (df['n_conjugatable_groups'] > 0)).sum()

stages = ['Input','Conjugatable','S_gating>50','Elevated+Conj.']
counts = [n_total, n_conj, n_gate50, n_elevated]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(stages, counts, color=['#CCC','#CCC','#6A5ACD','#E86F2C'], edgecolor='white')
ax.set_ylabel('Molecules'); ax.set_yscale('log')
for i, v in enumerate(counts): ax.text(i, v*1.1, str(v), ha='center', fontweight='bold')
ax.set_title(f'Search-Space Reduction: {n_total} -> {n_elevated} -> Top 50')
plt.tight_layout(); plt.show()

## 2. 5D Score Exploration

The 5D scoring formula: 5D = 0.35*S_gating + 0.25*S_chemistry + 0.15*S_novelty + 0.10*S_safety + 0.15*S_conjugate

In [ ]:
score_cols = ['S_gating','S_chemistry','S_novelty','S_safety','S_conjugate']
fig, axes = plt.subplots(1, 5, figsize=(14, 3))
colors = ['#6A5ACD','#27AE60','#2E86AB','#95A5A6','#E86F2C']
for ax, col, c, name in zip(axes, score_cols, colors, ['Gating','Chemistry','Novelty','Safety','Conjugate']):
    ax.hist(df[col].dropna(), bins=40, color=c, alpha=0.7, edgecolor='white', linewidth=0.3)
    ax.set_title(name, fontsize=9); ax.set_xlabel('Score')
plt.suptitle('5D Score Distributions', fontweight='bold')
plt.tight_layout(); plt.show()

## 3. Top 50 Candidates

49/50 contain single aliphatic OH handles, carbonate ester compatible.

In [ ]:
print('=== Top 10 Candidates ===')
display_cols = ['Total_Score_5D','S_gating','S_conjugate','MW','cLogP','dominant_linkage','conjugatable_types']
display(top50[display_cols].head(10).style.background_gradient(subset=['Total_Score_5D'], cmap='YlOrRd'))

## 4. TCM vs Synthetic Enrichment

TCM natural products show 8.5x higher mean S_gating than synthetic drugs.

In [ ]:
tcm = df[df['source']=='tcm_focused']['S_gating']
syn = df[df['source']=='round1']['S_gating']

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(syn, bins=50, alpha=0.5, density=True, color='#CCC', label=f'Synthetic (mean={syn.mean():.1f})')
ax.hist(tcm, bins=30, alpha=0.7, density=True, color='#27AE60', label=f'TCM NPs (mean={tcm.mean():.1f})')
ax.axvline(x=50, color='#E86F2C', linestyle='--', label='S_gating=50')
ax.set_xlabel('S_gating'); ax.set_ylabel('Density'); ax.legend()
ax.set_title(f'TCM vs Synthetic: {tcm.mean()/syn.mean():.1f}x enrichment')
plt.tight_layout(); plt.show()

## 5. Load Pre-trained Model

Use the trained RandomForest model for custom screening.

In [ ]:
import joblib
model = joblib.load('data/gatescore_RF_v1.joblib')
print(f'Model loaded: {type(model).__name__}')
print(f'Has predict_proba: {hasattr(model, "predict_proba")}')

## Citation

Wu Z. GateScore-Lipid: A Conjugatability-Aware Computational Protocol for Prioritizing Candidate Small-Molecule Modules in Brain-Targeted Lipid Nanoparticle Design. *Briefings in Bioinformatics* (2026).

## License
CC-BY 4.0